## part 1 Reward Engine
Mechanics of reward_engine.pyIn DeepSeek-R1-Zero, the reward function is 100% deterministic and rule-based (Equation 4: $R_{\text{rule}} = R_{\text{acc}} + R_{\text{fmt}}$).  To prevent zero-gradient deadlocks on a 135M model, our engine implements a 4-tier reward hierarchy:                       [ Generated Completion ]
                                  │
          ┌───────────────────────┴───────────────────────┐
          ▼                                               ▼
  [ Format Verification ]                         [ Accuracy Verification ]
  • Check <think>...</think>                      • Extract answer string
  • Check <answer>...</answer>                    • Parse via SymPy / Regex
  • Score: R_fmt in [0.0, 0.1]                    • Score: R_acc in [0.0, 1.0]
Strict Format Match ($R_{\text{fmt}} = 0.1$):The output strictly matches <think>...</think><answer>...</answer>.Partial / Soft Format Credit ($R_{\text{fmt}} \in (0.0, 0.1)$):If a small model outputs <think> or </think> or <answer> tags, but hasn't perfected the complete closing sequence yet, we give small partial credit ($+0.02$ to $+0.05$). This gives GRPO early positive reward variance so $\text{std}(\mathbf{r}) > 0$, allowing the policy to start learning structural formatting gradients from step 1.Accuracy Match via SymPy ($R_{\text{acc}} = 1.0$):We extract the content inside <answer>...</answer> and evaluate it symbolically against the ground truth answer using sympy. If $\text{simplify}(\text{prediction} - \text{truth}) == 0$, $R_{\text{acc}} = 1.0$.Fallback Raw String Match:If SymPy parsing fails due to formatting noise, it falls back to normalized numeric string comparison.

In [1]:
import subprocess
import sys
from pathlib import Path

repo_dir = Path.cwd()
engine_path = repo_dir / "reward_engine.py"

if not engine_path.exists():
    raise FileNotFoundError(f"Could not find reward_engine.py at {engine_path}")

result = subprocess.run(
    [sys.executable, str(engine_path)],
    cwd=repo_dir,
    capture_output=True,
    text=True,
)

print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise SystemExit(result.returncode)


=== Running Unit Test Suite for reward_engine.py ===
Test 1 [Perfect Output (Correct + Strict Format)]: PASSED
  Result -> Reward: 1.100 | Correct: True | Extracted: '30'
Test 2 [SymPy Equivalence (Fraction vs Decimal)]: PASSED
  Result -> Reward: 1.100 | Correct: True | Extracted: '0.5'
Test 3 [Wrong Answer + Strict Format]: PASSED
  Result -> Reward: 0.100 | Correct: False | Extracted: '99'
Test 4 [Partial Tags (Soft Format Credit)]: PASSED
  Result -> Reward: 0.025 | Correct: False | Extracted: '<think> 14 * 3 = 42, Answer is 30'
Test 5 [No Tags At All]: PASSED
  Result -> Reward: 0.000 | Correct: False | Extracted: 'The result is 30'

Unit Test Suite Summary: ALL TESTS PASSED!

